<a href="https://colab.research.google.com/github/SamRB-dev/Drone-Detection/blob/main/notebooks/Drone_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# Installation
#!pip install kagglehub

# Imports
import kagglehub, os
from collections import Counter
from pathlib import Path

# Variables
path = kagglehub.dataset_download("simeonajakwe/visiodect-for-scenario-based-multi-drone-detection")

Using Colab cache for faster access to the 'visiodect-for-scenario-based-multi-drone-detection' dataset.


In [22]:
path

'/kaggle/input/visiodect-for-scenario-based-multi-drone-detection'

# Dataset Exploration

In [23]:
# Count files by extension and top-level dataset folder.
extensions = Counter()
folders = Counter()

print("DATASET STRUCTURE\n")

for root, dirs, files in os.walk(path):
    # Show only the first three directory levels.
    level = os.path.relpath(root, path).count(os.sep) + 1
    if root == path:
        level = 0

    if level <= 3:
        print("    " * level + os.path.basename(root) + "/")

        # Display up to three filenames per directory as examples.
        for file in files[:3]:
            print("    " * (level + 1) + file)

        if len(files) > 3:
            print("    " * (level + 1) + f"... {len(files) - 3} more files")

    # Count every file, including those in deeper directories.
    for file in files:
        extensions[os.path.splitext(file)[1].lower()] += 1

        # Group files by their top-level dataset folder.
        relative = os.path.relpath(root, path)
        top_folder = relative.split(os.sep)[0]
        folders[top_folder] += 1

print("\nFILE COUNTS BY EXTENSION")
for ext, count in extensions.most_common():
    print(f"{ext or 'No extension'}: {count}")

print("\nFILE COUNTS BY TOP-LEVEL FOLDER")
for folder, count in folders.most_common():
    print(f"{folder}: {count}")


DATASET STRUCTURE

visiodect-for-scenario-based-multi-drone-detection/
    VisioDECT Dataset Upload/
        DJIFPV/
            labels/
            images/
        Mavic_Enterprise/
            labels/
            images/
        Anafi-Extended/
            labels/
            images/
        EFT-E410S/
            labels/
            images/
        DJIPhantom/
            labels/
            images/
        Mavic_Air/
            labels/
            images/

FILE COUNTS BY EXTENSION
.jpg: 20924
.xml: 19428
.txt: 19424
.csv: 17
.xlsx: 1

FILE COUNTS BY TOP-LEVEL FOLDER
VisioDECT Dataset Upload: 59794


In [28]:
# Locate the main dataset directory.
base = Path(path) / "VisioDECT Dataset Upload"

# Find all JPEG images recursively.
images = list(base.rglob("*.jpg"))

# Check whether any images were found.
print("Images found:", len(images))

if images:
    # Select the first image.
    image = images[0]

    # Find its corresponding annotation directory.
    label_dir = image.parent.parent / "labels"

    # Construct matching XML and TXT filenames.
    xml_file = label_dir / f"{image.stem}.xml"
    txt_file = label_dir / f"{image.stem}.txt"

    print("\nIMAGE:", image)
    print("XML EXISTS:", xml_file.exists())
    print("TXT EXISTS:", txt_file.exists())

    # Display the annotation contents, if available.
    if xml_file.exists():
        print("\nXML ANNOTATION:")
        print(xml_file.read_text()[:2000])

    if txt_file.exists():
        print("\nTXT ANNOTATION:")
        print(txt_file.read_text()[:1000])

else:
    print("No JPEG images found. Check the dataset directory.")

Images found: 20924

IMAGE: /kaggle/input/visiodect-for-scenario-based-multi-drone-detection/VisioDECT Dataset Upload/Mavic_Air/images/Sunny/MAVIC_Air_Sunny (173).jpg
XML EXISTS: False
TXT EXISTS: False


In [30]:
# Select the first image.
image = images[0]

# The image folder is structured as:
# DroneModel/images/Scenario/image.jpg
# Its parent is "images", and its parent is the drone model.
drone_folder = image.parent.parent

# Get the scenario name (Sunny, Cloudy or Evening).
scenario = image.parent.name

# Find the matching labels folder.
label_dir = drone_folder.parent / "labels" / scenario

# Build the expected annotation paths.
xml_file = label_dir / f"{image.stem}.xml"
txt_file = label_dir / f"{image.stem}.txt"

print("IMAGE:", image)
print("DRONE MODEL:", drone_folder.parent.name)
print("SCENARIO:", scenario)
print("LABEL DIRECTORY:", label_dir)
print("\nXML EXISTS:", xml_file.exists())
print("TXT EXISTS:", txt_file.exists())

# Display annotation contents if found.
if xml_file.exists():
    print("\nXML ANNOTATION:")
    print(xml_file.read_text()[:2000])

if txt_file.exists():
    print("\nTXT ANNOTATION:")
    print(txt_file.read_text()[:1000])


IMAGE: /kaggle/input/visiodect-for-scenario-based-multi-drone-detection/VisioDECT Dataset Upload/Mavic_Air/images/Sunny/MAVIC_Air_Sunny (173).jpg
DRONE MODEL: Mavic_Air
SCENARIO: Sunny
LABEL DIRECTORY: /kaggle/input/visiodect-for-scenario-based-multi-drone-detection/VisioDECT Dataset Upload/Mavic_Air/labels/Sunny

XML EXISTS: False
TXT EXISTS: False


In [31]:

# Locate the image and label directories for this drone model.
drone_folder = image.parents[2]
image_dir = drone_folder / "images" / scenario
label_dir = drone_folder / "labels" / scenario

# List a few actual filenames from both directories.
image_files = list(image_dir.iterdir())
label_files = list(label_dir.iterdir()) if label_dir.exists() else []

print("IMAGE DIRECTORY:", image_dir)
print("LABEL DIRECTORY:", label_dir)

print("\nIMAGE FILES:")
for file in image_files[:5]:
    print(file.name)

print("\nLABEL FILES:")
for file in label_files[:10]:
    print(file.name)

print("\nLABEL DIRECTORY EXISTS:", label_dir.exists())


IMAGE DIRECTORY: /kaggle/input/visiodect-for-scenario-based-multi-drone-detection/VisioDECT Dataset Upload/Mavic_Air/images/Sunny
LABEL DIRECTORY: /kaggle/input/visiodect-for-scenario-based-multi-drone-detection/VisioDECT Dataset Upload/Mavic_Air/labels/Sunny

IMAGE FILES:
MAVIC_Air_Sunny (173).jpg
MAVIC_Air_Sunny (510).jpg
MAVIC_Air_Sunny (273).jpg
MAVIC_Air_Sunny (22).jpg
MAVIC_Air_Sunny (1020).jpg

LABEL FILES:

LABEL DIRECTORY EXISTS: False


In [33]:

# Get the Mavic_Air directory.
drone_folder = image.parents[2]

# Print all directories and a few example files.
for folder in sorted(drone_folder.rglob("*")):
    if folder.is_dir():
        print(f"\n📁 {folder.relative_to(drone_folder)}")

        # Show up to three files in each directory.
        for file in list(folder.iterdir())[:3]:
            if file.is_file():
                print(f"    {file.name}")



📁 images

📁 images/Cloudy
    MAVIC_Air_Cloudy (675).jpg
    MAVIC_Air_Cloudy (201).jpg
    MAVIC_Air_Cloudy (874).jpg

📁 images/Evening
    MAVIC_Air_Evening (1003).jpg
    MAVIC_Air_Evening (964).jpg
    MAVIC_Air_Evening (245).jpg

📁 images/Sunny
    MAVIC_Air_Sunny (173).jpg
    MAVIC_Air_Sunny (510).jpg
    MAVIC_Air_Sunny (273).jpg

📁 labels

📁 labels/cloudy
    csv.csv

📁 labels/cloudy/txt
    MAVIC_Air_Cloudy (452).txt
    MAVIC_Air_Cloudy (1178).txt
    MAVIC_Air_Cloudy (93).txt

📁 labels/cloudy/voc
    MAVIC_Air_Cloudy (981).xml
    MAVIC_Air_Cloudy (262).xml
    MAVIC_Air_Cloudy (1059).xml

📁 labels/evening
    csv.csv

📁 labels/evening/txt
    MAVIC_Air_Evening (298).txt
    MAVIC_Air_Evening (671).txt
    MAVIC_Air_Evening (732).txt

📁 labels/evening/voc
    MAVIC_Air_Evening (723).xml
    MAVIC_Air_Evening (528).xml
    MAVIC_Air_Evening (806).xml

📁 labels/sunny
    csv.csv

📁 labels/sunny/txt
    MAVIC_Air_Sunny (752).txt
    MAVIC_Air_Sunny (391).txt
    MAVIC_Air_Sun

In [34]:

# Use the image selected earlier.
image = images[0]

# Identify its drone model and scenario.
drone_folder = image.parents[2]
scenario = image.parent.name.lower()

# Locate the corresponding TXT and XML annotation directories.
txt_dir = drone_folder / "labels" / scenario / "txt"
xml_dir = drone_folder / "labels" / scenario / "voc"

# Construct the annotation filenames using the image name.
txt_file = txt_dir / f"{image.stem}.txt"
xml_file = xml_dir / f"{image.stem}.xml"

# Check whether the corresponding annotations exist.
print("IMAGE:", image.name)
print("TXT EXISTS:", txt_file.exists())
print("XML EXISTS:", xml_file.exists())

# Display the TXT annotation.
if txt_file.exists():
    print("\nTXT CONTENT:")
    print(txt_file.read_text()[:1000])

# Display the XML annotation.
if xml_file.exists():
    print("\nXML CONTENT:")
    print(xml_file.read_text()[:2000])


IMAGE: MAVIC_Air_Sunny (173).jpg
TXT EXISTS: True
XML EXISTS: True

TXT CONTENT:
5 0.362013 0.831169 0.034091 0.050505

XML CONTENT:
<annotation>
	<folder>my-project-name</folder>
	<filename>MAVIC_Air_Sunny (173).jpg</filename>
	<path>/my-project-name/MAVIC_Air_Sunny (173).jpg</path>
	<source>
		<database>Unspecified</database>
	</source>
	<size>
		<width>1920</width>
		<height>1080</height>
		<depth>3</depth>
	</size>
	<object>
		<name>MAVIC_Air_Sunny</name>
		<pose>Unspecified</pose>
		<truncated>Unspecified</truncated>
		<difficult>Unspecified</difficult>
		<bndbox>
			<xmin>662</xmin>
			<ymin>870</ymin>
			<xmax>728</xmax>
			<ymax>925</ymax>
		</bndbox>
	</object>
</annotation>


In [35]:
# Count how frequently each class ID appears across all TXT annotations.
class_counts = Counter()
empty_labels = 0
total_labels = 0

for txt_file in Path(path).rglob("*.txt"):
    # Read each annotation file.
    lines = txt_file.read_text().strip().splitlines()

    # Empty files contain no annotated objects.
    if not lines or not lines[0].strip():
        empty_labels += 1
        continue

    # YOLO format: the first value on each line is the class ID.
    for line in lines:
        class_id = line.split()[0]
        class_counts[class_id] += 1
        total_labels += 1

print("OBJECT CLASS DISTRIBUTION")
for class_id, count in sorted(class_counts.items()):
    print(f"Class {class_id}: {count} objects")

print("\nTOTAL ANNOTATED OBJECTS:", total_labels)
print("EMPTY LABEL FILES:", empty_labels)


OBJECT CLASS DISTRIBUTION
Class 0: 1191 objects
Class 1: 1102 objects
Class 10: 1152 objects
Class 11: 1200 objects
Class 13: 1200 objects
Class 14: 989 objects
Class 15: 901 objects
Class 16: 1200 objects
Class 17: 1200 objects
Class 2: 1132 objects
Class 3: 1165 objects
Class 4: 1163 objects
Class 5: 1191 objects
Class 6: 1112 objects
Class 7: 1127 objects
Class 8: 1200 objects
Class 9: 1200 objects

TOTAL ANNOTATED OBJECTS: 19425
EMPTY LABEL FILES: 0


In [36]:

from collections import defaultdict, Counter
import xml.etree.ElementTree as ET

# Map each TXT class ID to the object names found in matching XML files.
class_names = defaultdict(Counter)

# Search only the actual YOLO annotation directories.
for txt_file in Path(path).rglob("labels/*/txt/*.txt"):

    # Find the matching XML annotation.
    xml_file = txt_file.parent.parent / "voc" / f"{txt_file.stem}.xml"

    if not xml_file.exists():
        continue

    # Read the class IDs from the TXT file.
    class_ids = [line.split()[0] for line in txt_file.read_text().splitlines()
                 if line.strip()]

    # Read the object names from the XML file.
    root = ET.parse(xml_file).getroot()
    names = [obj.findtext("name", default="Unknown")
             for obj in root.findall("object")]

    # Associate each class ID with the XML object name.
    for class_id, name in zip(class_ids, names):
        class_names[class_id][name] += 1

# Display the results.
print("CLASS ID → XML OBJECT NAME\n")

for class_id in sorted(class_names, key=int):
    print(f"Class {class_id}: {dict(class_names[class_id])}")


CLASS ID → XML OBJECT NAME

Class 0: {'MAVIC2_Enterprise_Dual_Cloudy': 893, 'Mavic_Enterprise_Cloudy': 298}
Class 1: {'MAVIC2_Enterprise_Dual_Evening': 623, 'Mavic_Enterprise_Evening': 281, 'MAVIC2_Enterprise_Dual_Cloudy': 198}
Class 2: {'MAVIC2_Enterprise_Dual_Sunny': 819, 'Mavic_Enterprise_Sunny': 291, 'MAVIC2_Enterprise_Dual_Cloudy': 22}
Class 3: {'Mavic_Air_Cloudy': 290, 'MAVIC_Air_Cloudy': 875}
Class 4: {'Mavic_Air_Evening': 296, 'MAVIC_Air_Evening': 867}
Class 5: {'MAVIC_Air_Sunny': 891, 'Mavic_Air_Sunny': 300}
Class 7: {'DJI_FPV_Evening': 1127}
Class 8: {'DJI_FPV_Sunny': 1200}
Class 9: {'EFTE410S_Cloudy': 1200}
Class 10: {'EFTE410S_Evening': 1152}
Class 11: {'EFTE410S_Sunny': 1200}
Class 13: {'Anafi_Extended_Evening': 1200}
Class 14: {'Anafi_Extended_Sunny': 989}
Class 15: {'DJI_Phantom_Cloudy': 901}
Class 16: {'DJI_Phantom_Evening': 1200}
Class 17: {'DJI_Phantom_Sunny': 1200}
